# 00 · Shared config
Shared constants for the climate-exposure example. Every notebook runs this with `%run ./00_config`.
Change the widgets (or the defaults below) to point the demo at a different catalog, schema or volume.

In [ ]:
dbutils.widgets.text("catalog", "main", "Catalog")
dbutils.widgets.text("schema", "climate_exposure", "Schema")
dbutils.widgets.text("volume", "raw", "Volume")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
VOLUME = dbutils.widgets.get("volume")
FQ = f"{CATALOG}.{SCHEMA}"                                 # table prefix
VOL = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"              # UC Volume root

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FQ}")              # the catalog must already exist (see README)
spark.sql(f"CREATE VOLUME IF NOT EXISTS {FQ}.{VOLUME}")

# ── Climate model selection (BARPA-R, downscaled CMIP6, BoM / Australian Climate Service) ──
BARPA_BUCKET = "https://bom-opendata-climate.s3.ap-southeast-2.amazonaws.com"
BARPA_PREFIX = "BARPA/CORDEX-CMIP6/DD/AUS-20i/BOM/ACCESS-CM2/{scenario}/r4i1p1f1/BARPA-R1-NN/v1-r1/mon/{var}/v20231001/"
VARIABLES = ["tasmax", "pr"]                                # daily max temperature, precipitation
BASELINE = ("historical", "200101-201012")                  # 2001-2010 baseline decade
FUTURES = {                                                 # scenario -> decades to compare
    "ssp126": ["204101-205012", "209101-210012"],
    "ssp370": ["204101-205012", "209101-210012"],
    "ssp585": ["204101-205012", "209101-210012"],
}
PERIOD_LABEL = {"200101-201012": "2001-2010", "204101-205012": "2041-2050", "209101-210012": "2091-2100"}

# ── Australian Bureau of Statistics (ASGS 2021 boundaries + Estimated Resident Population) ──
ABS_ASGS = ("https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/"
            "edition-3-july-2021-june-2026/access-and-downloads/digital-boundary-files")
ABS_FILES = {
    "SA2_2021_AUST_SHP_GDA2020.shp.zip": f"{ABS_ASGS}/SA2_2021_AUST_SHP_GDA2020.zip",
    "STE_2021_AUST_SHP_GDA2020.shp.zip": f"{ABS_ASGS}/STE_2021_AUST_SHP_GDA2020.zip",
    "erp_sa2_2001_2025.xlsx": "https://www.abs.gov.au/statistics/people/population/regional-population/2024-25/32180DS0003_2001-25.xlsx",
}

AUS_BBOX = "112,-44.5,154.5,-9.5"                          # lon/lat crop of the AUS-20i domain
H3_RES = 4                                                  # ~1,770 km² hexes ≈ 4 BARPA pixels each
HOT_MONTH_C = 35.0                                          # "hot month" = monthly-mean daily max ≥ 35 °C

print(f"Target: {FQ}   Volume: {VOL}")